# Anonymisation: comparing the three detectors

This notebook scores the three PII detectors of `itsm.privacy` against each
other on a sample of annotated tickets, calling the same library the CLI
(`itsm privacy eval`) and the API call:

| Backend | What it is | Needs |
|---|---|---|
| `rules` | Regular expressions, contextual triggers and an authored gazetteer | nothing |
| `transformer` | A multilingual Hugging Face NER model | `--extra hf` |
| `presidio` | Microsoft Presidio's `AnalyzerEngine` (spaCy + pattern recognisers) | `--extra presidio` + a spaCy model |

## Every backend runs alone

The one methodological decision worth stating up front: every backend is built
with `with_structural=False`, so none of them has the rule engine's structured
layer underneath.

That is not how they are deployed - the `transformer` backend composes with the
rules by default, and it is the better deployment - but **a composed detector
cannot be benchmarked**. Its score is the union of two systems: a gap in one is
invisible while the other covers it, so the table would compare pairs rather
than tools. Isolated, the comparison answers the question it is actually being
asked: *what does each of these find on its own?*

The same switch exists on the command line:

```bash
uv run itsm privacy eval --backend rules --backend transformer --backend presidio --isolate
```

## Before running

The dataset must exist and must carry the `pii_spans` ground truth - today that
means the generated one. The optional backends are skipped gracefully if their
extra is not installed, so the notebook runs end to end with `rules` alone.

```bash
uv run itsm data build                              # annotated dataset
uv sync --extra hf                                  # backend B
uv sync --extra presidio                            # backend C
uv run python -m spacy download en_core_web_lg      # backend C, one-time
```

In [ ]:
from __future__ import annotations

import json
import time

import pandas as pd

from itsm.config import get_settings
from itsm.pipelines.anonymise import load_raw_tickets
from itsm.privacy.anonymiser import Anonymiser
from itsm.privacy.evaluate import (
    entity_distribution,
    residual_pii_rate,
    score_detection,
    spans_from_payload,
)

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

settings = get_settings()

#: Free-text column carrying the ground truth. `description` is the long one -
#: more personal data per ticket, and the realistic worst case.
COLUMN = 'description'
#: Kept small on purpose: the transformer and Presidio are seconds per hundred
#: tickets on CPU. Raise it once the notebook runs.
SAMPLE = 500
SEED = 42

print('dataset:', settings.dataset, '| column:', COLUMN, '| sample:', SAMPLE)

## 1. The sample and its ground truth

Anonymisation cannot be scored without knowing where the personal data was. The
generator plants the identifiers itself and records the **character offsets** of
every one in the `pii_spans` column, which is what makes every number below a
measurement rather than an impression.

The support per entity type matters when reading the per-entity table later: a
recall of 0.5 on four annotated values is noise, not a finding.

In [ ]:
tickets = load_raw_tickets(settings)

if 'pii_spans' not in tickets.columns:
    raise RuntimeError(
        'this dataset carries no pii_spans annotation, so detection quality cannot '
        'be scored. Generate the annotated dataset with `uv run itsm data build`, '
        'or annotate a few hundred tickets of the real extract by hand.'
    )

sample = tickets.sample(n=min(SAMPLE, len(tickets)), random_state=SEED).reset_index(drop=True)
texts = sample[COLUMN].fillna('').astype(str).tolist()
references = [
    spans_from_payload(json.loads(payload or '{}'), COLUMN) for payload in sample['pii_spans']
]

n_values = sum(len(spans) for spans in references)
print(len(sample), 'tickets |', n_values, 'annotated personal values in', COLUMN)

support = pd.Series(entity_distribution(references), name='annotated values')
support.to_frame()

## 2. Build the three detectors, isolated

`Settings.pii_model()` picks the right kind of model per backend - a Hugging
Face checkpoint for `transformer`, a spaCy pipeline for `presidio`, nothing for
`rules`. Handing one to the other is a silent, confusing failure, which is why
the choice lives in one place.

In [ ]:
BACKENDS = ('rules', 'transformer', 'presidio')


def build(backend: str) -> Anonymiser:
    '''One detector, on its own recognisers.

    `with_structural=False` is what makes this a comparison of tools rather than
    of unions of tools. It is not the deployment configuration.
    '''
    return Anonymiser(
        backend=backend,
        model_name=settings.pii_model(backend),
        language=settings.presidio_language,
        with_structural=False,
    )

## 3. Run them

Each backend gets a warm-up call before the clock starts, so the timing measures
**throughput and not model download or load**. A backend whose extra is missing
is recorded in `skipped` and left out of the comparison rather than failing the
notebook.

In [ ]:
runs: dict[str, dict] = {}
skipped: dict[str, str] = {}

for backend in BACKENDS:
    try:
        anonymiser = build(backend)
        # Warm up: loads the model, so the measured time is inference only.
        anonymiser.anonymise_text('Merci de contacter Jean Dupont au 06 12 34 56 78.')
        started = time.perf_counter()
        outcome = anonymiser.anonymise_column(texts)
        elapsed = time.perf_counter() - started
    except (RuntimeError, ImportError, OSError) as error:
        skipped[backend] = str(error).splitlines()[0]
        print('skipped', backend, '->', skipped[backend])
        continue

    runs[backend] = {
        'spans': outcome.spans,
        'anonymised': outcome.anonymised,
        'seconds': elapsed,
    }
    print(backend, '->', outcome.n_spans, 'spans in', round(elapsed, 1), 's')

if not runs:
    raise RuntimeError('no backend could run - check the install notes at the top')

In [ ]:
scores = {
    backend: {
        'detection': score_detection(references, run['spans'], texts=texts),
        'residual': residual_pii_rate(run['anonymised'], references),
    }
    for backend, run in runs.items()
}

list(scores)

## 4. The headline comparison

Two families of number, and they answer different questions:

* **Span metrics** (`recall`, `precision`, `leak rate`) measure the *detector*.
* **`residual values in output`** measures the *deliverable*: the original
  values are searched for in the rewritten text. It is the only row that cannot
  be gamed by generous span matching, and the one to put in front of a reviewer.

`partial residuals` is the row people forget: `Jean Dupont` redacted from
`Dupont` onwards leaves `Jean` behind. Span arithmetic scores that as a hit; a
reader of the output still knows who it is.

In [ ]:
def headline(backend: str) -> dict[str, float]:
    detection = scores[backend]['detection']
    residual = scores[backend]['residual']
    recall = detection['recall_coverage']
    precision = detection['precision_coverage']
    f1 = 2 * recall * precision / (recall + precision) if recall + precision else 0.0
    return {
        'coverage recall': round(recall, 4),
        'exact-boundary recall': round(detection['recall_strict'], 4),
        'precision': round(precision, 4),
        'F1': round(f1, 4),
        'leak rate, direct ids': round(detection['leak_rate_direct_identifiers'], 4),
        'residual values in output': residual['residual_values'],
        'partial residuals': residual['residual_partial_values'],
        'false-positive spans': detection['false_positive_spans'],
        'over-redacted chars %': round(100 * detection['over_redacted_char_share'], 3),
        'seconds per 1k tickets': round(1000 * runs[backend]['seconds'] / len(texts), 1),
    }


comparison = pd.DataFrame({backend: headline(backend) for backend in runs})
comparison

## 5. Where each one actually fails

The headline hides the interesting part. Recall per entity type is where the
three separate, and the expected shape is:

* **formats** (`EMAIL`, `IBAN`, `CREDIT_CARD`, `IP_ADDRESS`, `PHONE`) - the rule
  engine and Presidio should both be near 1.0, the NER model near 0 on its own,
  because it was never trained on them;
* **`PERSON`** - the reverse, and the only entity where the model-based backends
  earn their dependency;
* **local identifiers** (`EMPLOYEE_ID`, `NATIONAL_ID`, `ADDRESS`,
  `DATE_OF_BIRTH`) - only the rule engine knows a *matricule* or a NIR exists.
  Standalone Presidio has no recogniser for them and should be at 0.

**That last row is the point of the isolated benchmark.** Composed, it would be
invisible: the rule layer underneath would cover it and the table would say
Presidio found the matricule.

In [ ]:
per_entity = pd.DataFrame(
    {
        backend: {
            entity: metrics['recall']
            for entity, metrics in scores[backend]['detection']['per_entity'].items()
        }
        for backend in runs
    }
)
per_entity = per_entity.join(support).sort_values('annotated values', ascending=False)

# Green to red on recall makes the gaps readable at a glance. Styling needs
# jinja2; fall back to the plain frame when it is not installed.
columns = list(runs)
try:
    styled = per_entity.style.format('{:.3f}', subset=columns).background_gradient(
        cmap='RdYlGn', vmin=0, vmax=1, subset=columns
    )
except ImportError:
    styled = per_entity.round(3)

styled

## 6. The number that actually decides

**The table above is rigged in the rule engine's favour, and saying so is not
enough.** The generator samples names from `itsm/data/catalog.py`, and the
gazetteer is built from the same lists, so its `PERSON` recall is near-perfect
*by construction*. A real corpus is full of names no list contains.

`unseen_name_benchmark()` re-runs every backend on twelve invented names
spanning naming traditions a French/English list will never cover - Krzysztof
Nowak, Oluwaseun Adeyemi, Mei Ling Chen, Yuki Nakamura - in corpus-like
sentences, half with a trigger word and half without.

It is small and synthetic, so it proves a **direction** rather than a number.
The direction is the one that decides which backend to deploy.

In [ ]:
from itsm.pipelines.eval_privacy import unseen_name_benchmark

probe = unseen_name_benchmark(settings, tuple(runs), isolate=True)

pd.DataFrame(
    {
        backend: {
            'recall on unseen names': round(entry['recall'], 4),
            'names never caught': len(entry['names_never_caught']),
            'names partially caught': len(entry['names_partially_caught']),
        }
        for backend, entry in probe['backends'].items()
    }
)

In [ ]:
# Which names each backend never caught, in any sentence frame. This is the
# error analysis an engineer iterates on, not a metric.
for backend, entry in probe['backends'].items():
    print(backend, '->', entry['names_never_caught'] or 'caught every name')

## 7. Error analysis

The two lists worth reading in full: what was missed, and what was redacted that
should not have been. Over-redaction is the safer direction, but it is not free
- it eats the product vocabulary the classifier lives on.

In [ ]:
for backend in runs:
    detection = scores[backend]['detection']
    print('===', backend, '===')
    print('  missed:')
    for value, count in detection['top_misses'][:6]:
        print('    ', count, 'x', value)
    print('  false positives:')
    for value, count in detection['top_false_positives'][:6]:
        print('    ', count, 'x', value)
    print()

In [ ]:
# The end-to-end failure: values still readable in the anonymised output.
for backend in runs:
    residual = scores[backend]['residual']
    print(
        backend,
        '->',
        residual['residual_values'],
        'of',
        residual['checked_values'],
        'values survive | direct identifiers:',
        round(residual['residual_rate_direct_identifiers'], 4),
    )
    for item in residual['examples'][:5]:
        print('    ', item)
    print()

## 8. Cost

A detector is also a runtime bill. At service-desk volume the difference between
milliseconds and seconds per ticket decides whether anonymisation can run
synchronously at submit or has to be a batch job - which is an architecture
decision, not a tuning one.

In [ ]:
throughput = pd.DataFrame(
    {
        backend: {
            'seconds for the sample': round(run['seconds'], 2),
            'ms per ticket': round(1000 * run['seconds'] / len(texts), 2),
            'tickets per second': round(len(texts) / run['seconds'], 1),
            'hours for 1M tickets': round(run['seconds'] / len(texts) * 1_000_000 / 3600, 2),
        }
        for backend, run in runs.items()
    }
)
throughput

## 9. Save the comparison

Written next to the reports the CLI produces, so a run from this notebook and a
run from `itsm privacy eval --isolate` sit side by side.

The verbatim values are kept only because this is generated data. **On a real
extract, strip them** - a report file gets pasted into tickets and chat.

In [ ]:
report = {
    'column': COLUMN,
    'n_tickets': len(sample),
    'n_annotated_values': n_values,
    'seed': SEED,
    'isolated': True,
    'skipped_backends': skipped,
    'headline': comparison.to_dict(),
    'per_entity_recall': per_entity.to_dict(),
    'throughput': throughput.to_dict(),
    'unseen_name_probe': probe['backends'],
}

settings.ensure_dirs()
path = settings.reports_dir / 'privacy_backend_comparison.json'
path.write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
print('written to', path)

## How to read the result

Three questions, in the order they should be asked:

1. **Does anything survive?** `residual values in output`, and the direct
   identifiers in particular. One surviving name, mailbox, phone number, NIR,
   IBAN or card number is a breach; a surviving matricule is a quality issue.
   These are not the same row and should never be averaged together.
2. **Where does it fail, and does that matter here?** A backend at 0 on
   `EMPLOYEE_ID` is fine if the extract has none, and disqualifying if every
   third ticket quotes one.
3. **What does it cost?** The dependency, the download, the runtime, and the
   over-redaction eating classifier vocabulary.

## What this comparison is not

**It is not the deployment configuration.** Isolated is how the choice is made;
composed is what should run. The expected conclusion is not *pick one* but
*formats from the rule layer, names from whichever model wins the unseen-name
probe on your own data* - which is what the default `transformer` backend
already does, and what `PresidioRecognizer(with_structural=True)` does once
Presidio has been chosen.

**The `PERSON` numbers here are optimistic for `rules` and pessimistic for
everything else**, for the reason given in section 6. Trust the unseen-name
probe over the headline recall, and trust a few hundred hand-annotated tickets
from the real extract over both.

**One language at a time.** Presidio is configured for a single spaCy pipeline
(`ITSM_PRESIDIO_LANGUAGE`), and this corpus is FR/EN. Running the notebook twice
under `en` and `fr` is how you price that constraint rather than assume it away.

---

* `docs/07-privacy.md` - the three detectors in full, and the recorded results.
* `docs/adr/0002-anonymisation.md` - why anonymisation is a pipeline stage with
  interchangeable detectors, and why Presidio does not compose by default.
* `itsm privacy eval --isolate` - the same comparison from the command line,
  with the downstream utility cost added.